# 01 - Dataset download and raw data assembly

This notebook downloads the public datasets used by the hybrid RAG system and
places them in the `data/` directory expected by the rest of the project.

| Dataset | Role | Destination |
| --- | --- | --- |
| Deceptive Opinion Spam Corpus v1.4 | Classifier training | `data/op_spam_v1_4.csv` |
| Yelp Fake Review Dataset (Kaggle) | Classifier training | `data/new_yelp_sample.csv` |
| Fake Reviews Dataset (Kaggle) | Cross-domain evaluation | `data/external_data_full.csv`, `data/external_data.csv` |
| Labelled Yelp Dataset (Kaggle) | Imbalanced hotel evaluation | `data/external_hotels_reviews_sample.csv` |
| YelpNYC + YelpZIP | FAISS knowledge base | `data/for_vector_db/` |

Large files are intentionally excluded from Git. The notebooks only need the
processed artefacts described in `src/spam_detection/config.py`.

## Environment setup

In [ ]:
import os
import re
import subprocess
import urllib.request
import zipfile

import pandas as pd

from spam_detection.config import (
    CROSS_DOMAIN_FULL_PATH,
    DATA_DIR,
    HOTEL_REVIEWS_PATH,
    OPSPAM_RAW_PATH,
    PREPARED_DIR,
    PREPROCESSED_DIR,
    YELP_SAMPLE_RAW_PATH,
)

for directory in (DATA_DIR, PREPARED_DIR, PREPROCESSED_DIR):
    directory.mkdir(parents=True, exist_ok=True)

## Kaggle credentials

Kaggle downloads require a personal API token. Export the credentials before
starting Jupyter instead of hard-coding them in the notebook:

```bash
export KAGGLE_USERNAME="your-user"
export KAGGLE_KEY="your-key"
```

Alternatively place `kaggle.json` in `~/.kaggle/kaggle.json`. The repository
never stores credentials.

In [ ]:
KAGGLE_USERNAME = os.getenv("KAGGLE_USERNAME")
KAGGLE_KEY = os.getenv("KAGGLE_KEY")
KAGGLE_READY = bool(KAGGLE_USERNAME and KAGGLE_KEY)

if not KAGGLE_READY:
    print("Kaggle credentials are not set; Kaggle downloads will be skipped.")
else:
    print(f"Kaggle user: {KAGGLE_USERNAME}")


def download_kaggle_dataset(dataset: str, target: Path) -> None:
    """Download and unzip a Kaggle dataset when credentials are available."""
    if not KAGGLE_READY:
        print(f"Skipping {dataset}: KAGGLE_USERNAME/KAGGLE_KEY are not set.")
        return
    target.mkdir(parents=True, exist_ok=True)
    env = os.environ.copy()
    env["KAGGLE_USERNAME"] = KAGGLE_USERNAME
    env["KAGGLE_KEY"] = KAGGLE_KEY
    subprocess.run(
        ["kaggle", "datasets", "download", "-d", dataset, "-p", str(target), "--unzip"],
        check=True,
        env=env,
    )

## 1. Deceptive Opinion Spam Corpus v1.4

The corpus contains 1,600 hotel reviews written by humans or generated by
crowdsourcing. Files are named `t_<hotel>_<id>.txt` for truthful reviews and
`d_<hotel>_<id>.txt` for deceptive reviews.

In [ ]:
OP_URL = "https://myleott.com/op_spam_v1.4.zip"
OP_ROOT = DATA_DIR / "op_spam_v1.4"

if not OPSPAM_RAW_PATH.exists():
    archive = DATA_DIR / "op_spam_v1.4.zip"
    if not archive.exists():
        print("Downloading the Deceptive Opinion Spam Corpus...")
        urllib.request.urlretrieve(OP_URL, archive)
    with zipfile.ZipFile(archive) as handle:
        handle.extractall(DATA_DIR)

    rows = []
    for path in OP_ROOT.rglob("*.txt"):
        match = re.match(r"^([td])_([A-Za-z]+)_\d+\.txt$", path.name)
        if not match:
            continue
        text = path.read_text(encoding="utf-8", errors="ignore").strip()
        if not text:
            continue
        polarity = "positive" if "positive" in str(path).lower() else "negative"
        rows.append(
            {
                "text": text,
                "label": 0 if match.group(1) == "t" else 1,
                "polarity": polarity,
                "hotel": match.group(2),
            }
        )

    pd.DataFrame(rows).to_csv(OPSPAM_RAW_PATH, index=False)

df_opspam = pd.read_csv(OPSPAM_RAW_PATH)
print(df_opspam.shape)
print(df_opspam["label"].value_counts())

## 2. Yelp Fake Review Dataset

The Kaggle dataset provides `new_data_train.csv` and `new_data_test.csv`.
They are concatenated into `data/new_yelp_sample.csv` with the columns used by
the preprocessing pipeline (`reviewContent`, `flagged`).

In [ ]:
YELP_DIR = DATA_DIR / "yelp_fake_review"
if not YELP_SAMPLE_RAW_PATH.exists():
    download_kaggle_dataset("virenraina/yelp-fake-review-dataset", YELP_DIR)
    train_path = YELP_DIR / "new_data_train.csv"
    test_path = YELP_DIR / "new_data_test.csv"
    if train_path.exists() and test_path.exists():
        train = pd.read_csv(train_path, sep="\t")
        test = pd.read_csv(test_path, sep="\t")
        yelp = pd.concat([train, test], ignore_index=True)
        required = {"reviewContent", "flagged"}
        if not required.issubset(yelp.columns):
            raise ValueError(
                f"Unexpected Yelp columns: {list(yelp.columns)}. "
                f"Expected at least {sorted(required)}."
            )
        yelp.to_csv(YELP_SAMPLE_RAW_PATH, index=False)

if YELP_SAMPLE_RAW_PATH.exists():
    df_yelp = pd.read_csv(YELP_SAMPLE_RAW_PATH)
    print(df_yelp.shape)
    print(df_yelp["flagged"].value_counts(dropna=False))

## 3. Cross-domain and hotel evaluation sets

* `mexwell/fake-reviews-dataset` contains LLM-generated (`CG`) and original
  (`OR`) reviews across several product categories.
* `abidmeeraj/yelp-labelled-dataset` is heavily imbalanced and is used for the
  hotel-review scenario (90% genuine, 10% fake after sampling).

In [ ]:
EXTERNAL_DIR = DATA_DIR / "external"
if not CROSS_DOMAIN_FULL_PATH.exists():
    download_kaggle_dataset("mexwell/fake-reviews-dataset", EXTERNAL_DIR / "fake_reviews")
    candidates = list((EXTERNAL_DIR / "fake_reviews").glob("*.csv"))
    if candidates:
        external = pd.read_csv(candidates[0])
        external = external.rename(columns={"text_": "text"})
        external.to_csv(CROSS_DOMAIN_FULL_PATH, index=False)

if CROSS_DOMAIN_FULL_PATH.exists():
    external = pd.read_csv(CROSS_DOMAIN_FULL_PATH).rename(columns={"text_": "text"})
    external["label"] = external["label"].map({"CG": 1, "OR": 0})
    sample = external.sample(n=min(3000, len(external)), random_state=42)
    sample = sample[["category", "rating", "label", "text"]]
    sample.to_csv(DATA_DIR / "external_data.csv", index=False)
    print(sample["label"].value_counts())

In [ ]:
LABELLED_YELP = DATA_DIR / "Labelled_Yelp_Dataset.csv"
if not HOTEL_REVIEWS_PATH.exists():
    download_kaggle_dataset("abidmeeraj/yelp-labelled-dataset", EXTERNAL_DIR / "labelled_yelp")
    if not LABELLED_YELP.exists():
        matches = list((EXTERNAL_DIR / "labelled_yelp").glob("*.csv"))
        if matches:
            LABELLED_YELP = matches[0]

if LABELLED_YELP.exists():
    hotels = pd.read_csv(LABELLED_YELP)
    hotels = hotels.rename(columns={"Review": "text", "Label": "label"})
    hotels = hotels.sample(n=min(3000, len(hotels)), random_state=42)
    hotels["label"] = hotels["label"].map({1: 0, -1: 1})
    hotels[["User_id", "Product_id", "Rating", "Date", "text", "label"]].to_csv(
        HOTEL_REVIEWS_PATH, index=False
    )
    print(hotels["label"].value_counts(normalize=True))

## 4. YelpNYC and YelpZIP knowledge base

These two datasets are not redistributed here. Download them from the
original sources and place the files as follows:

```
data/for_vector_db/YelpNYC/Yelp_dataset.xlsx
data/for_vector_db/YelpNYC/Yelp_Metadata.xlsx
data/for_vector_db/yelpzip.csv
```

Notebook `03_build_vector_database.ipynb` merges, deduplicates and indexes
them. The released repository can also use the already prepared
`data/faiss/Yelp_NYC_ZIP.parquet` and `data/faiss/Yelp_NYC_ZIP_roberta_index.faiss`
if those files were obtained separately.

## Expected layout

```
data/
  op_spam_v1_4.csv
  new_yelp_sample.csv
  external_data_full.csv
  external_data.csv
  external_hotels_reviews_sample.csv
  cross_domain_10k.csv
  for_vector_db/
  faiss/
  prepared/
  preprocessed/
```